# GraphRAG-Bench — RAG génératif contrôlé sur Google Colab

**Objectif :** comparer **TF-IDF + Qwen2.5-3B-Instruct** et **BGE-large + le même Qwen2.5-3B-Instruct**, sur 12 questions, cinq passages par question et les mêmes paramètres de génération.

- Le générateur ne reçoit **jamais** les réponses officielles ni les preuves de référence ; seules la question et les cinq passages entrent dans le prompt.
- **GPU requis :** Tesla T4 ou GPU CUDA avec une capacité comparable ; modèle chargé en float16.
- **Aucune API LLM payante**. Google Colab gratuit reste limité en durée, quotas et disponibilité ; le modèle Qwen a une licence à consulter avant toute réutilisation hors recherche.
- Les sorties sont **des réponses générées**, pas des scores de qualité. L'annotation humaine des réponses est une étape séparée.

Exécute les cellules **dans l'ordre**. Une question pilote est d'abord traitée par les deux méthodes, puis les douze questions sont traitées. Chaque sortie est enregistrée de façon atomique après génération. Pour survivre à une réinitialisation de session, nous proposons une sauvegarde dans Google Drive (optionnelle).

## 1 — Vérifier le matériel

In [ ]:
import torch, psutil, shutil
print("RAM (Go):", round(psutil.virtual_memory().total / 1024**3, 2))
print("Disque libre (Go):", round(shutil.disk_usage("/content").free / 1024**3, 2))
print("CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Active Exécution → Modifier le type d'exécution → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (Go):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

## 2 — Choisir l'emplacement des résultats

Par défaut, les résultats restent dans le stockage **temporaire** Colab : télécharge le ZIP en fin de session. Pour assurer une reprise après réinitialisation du runtime, passe `SAUVEGARDE_DRIVE` à `True` et autorise explicitement le montage de ton Drive.

In [ ]:
from pathlib import Path
SAUVEGARDE_DRIVE = False  # Passer à True pour la sauvegarde persistante

if SAUVEGARDE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SORTIES = Path("/content/drive/MyDrive/GraphRAG-Bench-LLM")
else:
    SORTIES = Path("/content/graphrag_llm_results")
SORTIES.mkdir(parents=True, exist_ok=True)
print("Dossier des résultats:", SORTIES)
print("Persistance après réinitialisation:", SAUVEGARDE_DRIVE)

## 3 — Charger le script GPU T4 corrigé (BatchEncoding)

Si tu as déjà modifié le script dans Colab, une copie de sauvegarde du fichier est créée avant de récupérer uniquement le script et les tests validés dans GitHub. On ne modifie ni le reste du dépôt ni les fichiers de résultats. La session doit disposer d'un GPU CUDA avant la génération.

In [ ]:
import subprocess, sys, shutil
from pathlib import Path

BRANCHE = "feature/graphrag-pilot-llm-generation"
REPO = Path("/content/GraphRAG-Benchmark-LLM")
if not REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", BRANCHE,
        "https://github.com/MamadyIA/GraphRAG-Benchmark.git", str(REPO)
    ], check=True)
else:
    # Ne pas écraser aveuglément les autres modifications locales.
    # Récupérer uniquement les deux fichiers corrigés à partir du commit distant.
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1",
                    "origin", BRANCHE], check=True)
    for relative in [
        "experiments/pilot_llm_generate.py",
        "tests/test_pilot_llm_generate.py",
    ]:
        target = REPO / relative
        if target.exists():
            shutil.copy2(target, target.with_name(target.name + ".before_batchencoding_fix.bak"))
        file_bytes = subprocess.check_output(
            ["git", "-C", str(REPO), "show", "FETCH_HEAD:" + relative]
        )
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(file_bytes)
    print("Correctif BatchEncoding synchronisé sans modifier les autres fichiers.")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "accelerate>=1.0", "transformers>=4.45,<6"
], check=True)
print("Dépôt:", REPO)
print("CUDA:", __import__("torch").cuda.is_available())


## 4 — Réutiliser les deux fichiers de récupération

Le notebook recherche les fichiers dans `/content/graphrag_audit_results`. S'ils n'y sont plus, il te demande de téléverser **GraphRAG_Comparaison.zip** (l'archive sauvegardée lors de l'expérience précédente). Il extrait uniquement les deux JSON nécessaires ; aucun script ni archive inconnue n'est exécuté.

In [ ]:
from zipfile import ZipFile
from google.colab import files
INPUTS = Path("/content/graphrag_audit_results")
INPUTS.mkdir(parents=True, exist_ok=True)
NOMS = ("pilot_tfidf_shared_retrieval.json", "pilot_bge_shared_retrieval.json")

if not all((INPUTS / n).is_file() for n in NOMS):
    print("Téléverse GraphRAG_Comparaison.zip depuis ton ordinateur.")
    uploaded = files.upload()
    archive_names = [n for n in uploaded if n.lower().endswith(".zip")]
    if len(archive_names) != 1:
        raise ValueError("Une seule archive ZIP contenant les résultats est attendue.")
    with ZipFile(archive_names[0]) as z:
        for filename in NOMS:
            matches = [name for name in z.namelist() if name == filename]
            if len(matches) != 1:
                raise ValueError("Fichier manquant ou ambigu: " + filename)
            (INPUTS / filename).write_bytes(z.read(filename))

sys.path.insert(0, str(REPO / "experiments"))
from pilot_llm_generate import paired_inputs
pairs = paired_inputs(INPUTS)
assert len(pairs) == 12
print("Comparaison prête:", len(pairs), "questions ;", len(pairs) * 2, "générations")

## 5 — Exécuter dix tests hors modèle puis valider les prompts

Les tests couvrent maintenant aussi la compatibilité des sorties **BatchEncoding** avec Transformers 5 et le respect de la limite de tokens. Les réponses et preuves officielles ne sont pas transmises au LLM.

In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover",
   "-s", "tests", "-p", "test_pilot_llm_generate.py", "-v"], cwd=REPO, check=True)

SCRIPT = str(REPO / "experiments/pilot_llm_generate.py")
ETAT = SORTIES / "pilot_qwen25_3b_answers.json"
REVUE = SORTIES / "pilot_qwen25_3b_review.csv"
BASE = [sys.executable, SCRIPT, "--retrieval-dir", str(INPUTS),
        "--output", str(ETAT), "--review", str(REVUE),
        "--model", "Qwen/Qwen2.5-3B-Instruct", "--device", "cuda",
        "--max-input-tokens", "3584", "--max-new-tokens", "384"]
subprocess.run(BASE + ["--dry-run"], check=True)
print("Validation de l'alignement et des prompts terminée.")

## 6 — Essai contrôlé sur une question (deux réponses)

Le modèle Qwen2.5-3B-Instruct est téléchargé la première fois et chargé sur le GPU T4. Le script génère **deux réponses** à la première question, l'une avec TF-IDF et l'autre avec BGE-large. Le fichier JSON est sauvegardé après chaque réponse. Les citations [P#] identifient des passages fournis, pas des sources vérifiées.

In [ ]:
subprocess.run(BASE + ["--limit-questions", "1"], check=True)
import json
etat = json.loads(ETAT.read_text(encoding="utf-8"))
print("Statut:", etat["status"], "Réponses générées:", len(etat["completed"]))
for key, row in etat["completed"].items():
    print("\n", key, "|", row["finish_reason"], "|", row["generated_tokens"], "tokens")
    print(row["generated_answer"][:900])

## 7 — Génération complète : 12 questions × 2 méthodes

Relance le même programme sans limite. Les deux réponses de la première question seront réutilisées **sans nouvelle génération**, et les autres seront enregistrées au fur et à mesure. En cas d'arrêt du processus, réexécute cette cellule avec les mêmes fichiers et le même modèle.

In [ ]:
subprocess.run(BASE, check=True)
etat = json.loads(ETAT.read_text(encoding="utf-8"))
assert etat["status"] == "COMPLETE", "Certaines réponses manquent."
assert len(etat["completed"]) == 24
print("QUALIFICATION TECHNIQUE: 24 réponses produites et sauvegardées.")
print("Feuille de revue humaine:", REVUE)

## 8 — Télécharger le dossier de résultats

La feuille CSV contient, pour chaque question, les deux réponses et la référence officielle **uniquement pour l'évaluation humaine**. Les champs de jugement sont laissés vides. Pour un travail scientifique, vérifier la fidélité aux passages et l'exactitude de chaque réponse avant de calculer des scores.

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files
archive = Path("/content/GraphRAG_LLM_Pilote.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for path in (ETAT, REVUE):
        if path.exists():
            z.write(path, arcname=path.name)
    for filename in NOMS:
        path = INPUTS / filename
        if path.exists():
            z.write(path, arcname=path.name)
print("Archive créée:", archive, "Mo:", round(archive.stat().st_size / 1024**2, 2))
files.download(str(archive))

## Limites et précautions

- Les 24 réponses sont produites **sans annotation humaine automatique de vérité**. Le questionnaire de génération créative exige une évaluation spécifique ; l'exactitude ne peut pas être réduite à un simple matching de chaînes.
- L'évaluation des preuves de récupération est actuellement **provisoire** et ne doit pas être présentée comme une mesure officielle.
- Le décodage glouton et les prompts identiques contrôlent les principales variables, mais une reproductibilité bit à bit n'est pas garantie entre matériels et versions.
- Le modèle Qwen2.5-3B-Instruct n'est pas nécessairement le LLM de référence de l'article. Il constitue un **comparatif additionnel et contrôlé**, pas une réplication exacte des scores officiels.
- Les modèles et notebooks utilisent le quota Colab gratuit : disponibilité, sessions et limites ne sont pas garanties.
- Ne pas inclure les données de vérité terrain dans un prompt d'évaluation. Les réponses et la revue sont stockées localement ou sur Drive uniquement à la demande.